# INR/USD Quantitative Forecasting & Trading Backtest

This notebook implements a state-of-the-art forecasting and trading backtest pipeline for the USD/INR exchange rate using **Weekly averages** (providing ~590 observations from 2015 to 2026). 

We compare **9 different models** across statistical, machine learning, regularized regression, and hybrid domains:
1. **Naïve (Random Walk)**: No change predicted.
2. **Simple Exponential Smoothing (SES)**: Fits level predictions on historical levels.
3. **ARIMA (Non-Seasonal)**: Classical univariate time-series model (best order determined on training set).
4. **ARIMAX**: Linear time-series regression with 1-week lagged exogenous macro features.
5. **Lasso Regression**: Regularized linear model designed to mitigate multicollinearity among macro drivers.
6. **Random Forest Regressor**: Non-linear ensemble model.
7. **Gradient Boosting Regressor**: Standalone non-linear sequential boosting model.
8. **ARIMA + Lasso Hybrid**: Hybrids linear ARIMA with Lasso regression predicting the residuals.
9. **ARIMA + Gradient Boosting Hybrid**: Hybrids linear ARIMA with Gradient Boosting predicting the residuals.

### Overcoming Core PITFALLS:
- **Spurious Regression**: Resolved by taking **first differences** (changes/returns) of all continuous macro indicators and exchange rates to achieve stationarity ($I(0)$).
- **Look-Ahead Bias**: Resolved by **lagging all exogenous features by 1 week** ($X_{t-1}$). To predict week $t$, we only use information known up to week $t-1$.
- **Realistic Validation**: We employ a **Rolling 1-Step-Ahead Validation** loop over a 200-week test window. Models are re-fit weekly, anchoring level predictions on the previous week's actual rate ($y_{t-1}$).


In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import SimpleExpSmoothing
from pmdarima import auto_arima
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
import json
import warnings
warnings.filterwarnings("ignore")


In [2]:
df = pd.read_csv("../data/processed/master_df.csv", index_col=0, parse_dates=True)
weekly = df[["USDINR","CRUDE","DXY","Rate_Spread","Geo_Tension"]].resample("W").mean()
weekly.dropna(inplace=True)
print(f"Weekly data shape: {weekly.shape[0]} weeks")


Weekly data shape: 595 weeks


In [3]:
weekly["USDINR_diff"] = weekly["USDINR"].diff()
weekly["CRUDE_diff"] = weekly["CRUDE"].diff()
weekly["DXY_diff"] = weekly["DXY"].diff()
weekly["Rate_Spread_diff"] = weekly["Rate_Spread"].diff()

exog_base = ["CRUDE_diff", "DXY_diff", "Rate_Spread_diff", "Geo_Tension"]
lagged_exog = weekly[exog_base].shift(1)
lagged_exog.columns = [c + "_lag1" for c in exog_base]

# Momentum & Calendar features
weekly["inr_mom_4w"] = weekly["USDINR_diff"].rolling(4).mean().shift(1)
weekly["inr_mom_12w"] = weekly["USDINR_diff"].rolling(12).mean().shift(1)
weekly["is_fiscal_yr_end"] = (weekly.index.month == 3).astype(float)
weekly["is_qtr_end"] = weekly.index.month.isin([3, 6, 9, 12]).astype(float)

features_list = [
    "CRUDE_diff_lag1", "DXY_diff_lag1", "Rate_Spread_diff_lag1", "Geo_Tension_lag1",
    "inr_mom_4w", "inr_mom_12w", "is_fiscal_yr_end", "is_qtr_end"
]

model_df = pd.concat([
    weekly[["USDINR", "USDINR_diff"]],
    lagged_exog,
    weekly[["inr_mom_4w", "inr_mom_12w", "is_fiscal_yr_end", "is_qtr_end"]]
], axis=1).dropna()
print(f"Model dataset shape: {model_df.shape}")


Model dataset shape: (582, 10)


In [4]:
TEST_WEEKS = 200
train = model_df.iloc[:-TEST_WEEKS]
test  = model_df.iloc[-TEST_WEEKS:]
print(f"Train size: {len(train)}, Test size: {len(test)}")


Train size: 382, Test size: 200


In [5]:
print("Finding optimal ARIMA order on training set...")
auto_model = auto_arima(
    train["USDINR_diff"],
    seasonal=False,
    stepwise=True,
    suppress_warnings=True,
    information_criterion="aic",
    max_p=5, max_q=5
)
order = auto_model.order
print(f"Best ARIMA order: {order}")


Finding optimal ARIMA order on training set...


Best ARIMA order: (1, 0, 0)


In [6]:
preds = {
    "naive": [], "es": [], "arima": [], "arimax": [], "lasso": [], "rf": [], "gb": [],
    "arima_lasso": [], "arima_gb": []
}

for i in range(TEST_WEEKS):
    if i % 25 == 0:
        print(f"Rolling forecast step {i}/{TEST_WEEKS}...")

    curr_train = model_df.iloc[:-(TEST_WEEKS - i)]
    curr_test = model_df.iloc[-(TEST_WEEKS - i):].iloc[0]

    y_train_diff = curr_train["USDINR_diff"]
    y_train_level = curr_train["USDINR"]
    X_train_exog = curr_train[features_list]
    prev_level = y_train_level.iloc[-1]

    test_exog_val = pd.DataFrame([curr_test[features_list]], columns=features_list)

    # 1. Naïve Baseline (No change predicted)
    preds["naive"].append(prev_level)

    # 2. Exponential Smoothing (fit on levels)
    try:
        ses_model = SimpleExpSmoothing(y_train_level).fit(optimized=True, use_brute=True)
        pred_es_level = ses_model.forecast(steps=1).iloc[0]
        preds["es"].append(pred_es_level)
    except Exception as e:
        preds["es"].append(prev_level)

    # 3. ARIMA (non-seasonal)
    arima_train_residuals = y_train_diff.copy()
    pred_arima_diff = 0.0
    try:
        model_arima = SARIMAX(
            y_train_diff,
            order=order,
            seasonal_order=(0,0,0,0),
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(disp=False)
        pred_arima_diff = model_arima.forecast(steps=1).iloc[0]
        preds["arima"].append(prev_level + pred_arima_diff)
        
        # In-sample residuals for hybrid models
        arima_fitted = model_arima.fittedvalues
        arima_train_residuals = y_train_diff - arima_fitted
    except Exception as e:
        preds["arima"].append(prev_level)

    # 4. ARIMAX
    try:
        model_arimax = SARIMAX(
            y_train_diff,
            exog=X_train_exog,
            order=order,
            seasonal_order=(0,0,0,0),
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(disp=False)
        pred_arimax_diff = model_arimax.forecast(steps=1, exog=test_exog_val).iloc[0]
        preds["arimax"].append(prev_level + pred_arimax_diff)
    except Exception as e:
        preds["arimax"].append(prev_level)

    # 5. Lasso Regression
    model_lasso = Lasso(alpha=0.001).fit(X_train_exog, y_train_diff)
    pred_lasso_diff = model_lasso.predict(test_exog_val)[0]
    preds["lasso"].append(prev_level + pred_lasso_diff)

    # 6. Random Forest
    model_rf = RandomForestRegressor(n_estimators=50, random_state=42).fit(X_train_exog, y_train_diff)
    pred_rf_diff = model_rf.predict(test_exog_val)[0]
    preds["rf"].append(prev_level + pred_rf_diff)

    # 7. Gradient Boosting
    model_gb = GradientBoostingRegressor(n_estimators=50, random_state=42).fit(X_train_exog, y_train_diff)
    pred_gb_diff = model_gb.predict(test_exog_val)[0]
    preds["gb"].append(prev_level + pred_gb_diff)

    # 8. ARIMA + Lasso Hybrid
    try:
        model_lasso_resid = Lasso(alpha=0.001).fit(X_train_exog, arima_train_residuals)
        pred_lasso_resid = model_lasso_resid.predict(test_exog_val)[0]
        pred_arima_lasso_diff = pred_arima_diff + pred_lasso_resid
        preds["arima_lasso"].append(prev_level + pred_arima_lasso_diff)
    except Exception as e:
        preds["arima_lasso"].append(prev_level + pred_arima_diff)

    # 9. ARIMA + Gradient Boosting Hybrid
    try:
        model_gb_resid = GradientBoostingRegressor(n_estimators=50, random_state=42).fit(X_train_exog, arima_train_residuals)
        pred_gb_resid = model_gb_resid.predict(test_exog_val)[0]
        pred_arima_gb_diff = pred_arima_diff + pred_gb_resid
        preds["arima_gb"].append(prev_level + pred_arima_gb_diff)
    except Exception as e:
        preds["arima_gb"].append(prev_level + pred_arima_diff)


Rolling forecast step 0/200...


Rolling forecast step 25/200...


Rolling forecast step 50/200...


Rolling forecast step 75/200...


Rolling forecast step 100/200...


Rolling forecast step 125/200...


Rolling forecast step 150/200...


Rolling forecast step 175/200...


In [7]:
y_test_actual = test["USDINR"].values
prev_test_levels = model_df["USDINR"].iloc[-(TEST_WEEKS+1):-1].values
actual_returns = (y_test_actual - prev_test_levels) / prev_test_levels

results = {}
trading_cum_rets = {}

def directional_accuracy(y_true, y_pred, y_prev):
    true_dir = np.sign(y_true - y_prev)
    pred_dir = np.sign(y_pred - y_prev)
    valid = true_dir != 0
    return np.mean(true_dir[valid] == pred_dir[valid]) * 100

naive_preds = np.array(preds["naive"])
naive_rmse = np.sqrt(mean_squared_error(y_test_actual, naive_preds))

for model_name, pred_levels in preds.items():
    pred_levels = np.array(pred_levels)

    # Core metrics
    mape = mean_absolute_percentage_error(y_test_actual, pred_levels) * 100
    rmse = np.sqrt(mean_squared_error(y_test_actual, pred_levels))
    theils_u = rmse / naive_rmse

    # Directional Accuracy
    mda = directional_accuracy(y_test_actual, pred_levels, prev_test_levels)

    # Backtest Simulation
    predicted_changes = pred_levels - prev_test_levels
    signals = np.sign(predicted_changes)
    strat_returns = signals * actual_returns
    
    # Cumulative returns
    cum_returns = np.cumprod(1 + strat_returns) - 1
    final_cum_ret = cum_returns[-1] * 100

    # Sharpe Ratio (Ann.) - Information Ratio (Rf = 0)
    # Epsilon guard: a constant array's std() is never exactly 0 in floating
    # point (~1e-19), so '!= 0' lets a near-zero denominator blow up to +/-1e16.
    EPS = 1e-8
    std0 = np.std(strat_returns)
    sharpe_rf0 = np.sqrt(52) * (np.mean(strat_returns) / std0) if std0 > EPS else 0.0

    # Sharpe Ratio adjusted for India 91-day T-bill (~6.5% annual, which is ~0.125% per week)
    rf_weekly = 0.065 / 52
    strat_returns_excess = strat_returns - rf_weekly
    std65 = np.std(strat_returns_excess)
    sharpe_rf6_5 = np.sqrt(52) * (np.mean(strat_returns_excess) / std65) if std65 > EPS else 0.0

    results[model_name] = {
        "MAPE (%)": mape,
        "RMSE": rmse,
        "Theil's U": theils_u,
        "MDA (%)": mda,
        "Sharpe Ratio (Rf=0)": sharpe_rf0,
        "Sharpe Ratio (Rf=6.5%)": sharpe_rf6_5,
        "Cumulative Return (%)": final_cum_ret
    }
    trading_cum_rets[model_name] = list(cum_returns)

results_df = pd.DataFrame(results).T
print("=== LEAGUE TABLE (200 WEEKS WITH HYBRIDS) ===")
print(results_df.to_string())


=== LEAGUE TABLE (200 WEEKS WITH HYBRIDS) ===
             MAPE (%)      RMSE  Theil's U  MDA (%)  Sharpe Ratio (Rf=0)  Sharpe Ratio (Rf=6.5%)  Cumulative Return (%)
naive        0.331104  0.401884   1.000000      0.0             0.000000           -4.156918e+16               0.000000
es           0.331104  0.401884   1.000000     43.0            -0.963610           -2.921717e+00             -11.766970
arima        0.328884  0.403015   1.002814     57.0             0.963610           -9.944973e-01              12.848352
arimax       0.338344  0.408549   1.016584     51.5             0.374835           -1.568640e+00               4.713933
lasso        0.329478  0.398798   0.992321     59.0             1.015006           -9.449810e-01              13.577147
rf           0.387529  0.446352   1.110648     54.5             0.282837           -1.659511e+00               3.484312
gb           0.359491  0.423654   1.054169     53.5             0.364940           -1.578399e+00               4.5

In [8]:
fig = go.Figure()
for model_name, cum_rets in trading_cum_rets.items():
    fig.add_trace(go.Scatter(
        x=[d.strftime('%Y-%m-%d') for d in test.index],
        y=[v * 100 for v in cum_rets],
        mode='lines',
        name=f"{model_name.upper()} (Sharpe: {results[model_name]['Sharpe Ratio (Rf=0)']:.2f})"
    ))

fig.update_layout(
    title="Out-of-Sample Trading Backtest: Cumulative Returns (200 Weeks)",
    xaxis_title="Date",
    yaxis_title="Cumulative Return (%)",
    template="plotly_white",
    legend_title="Models",
    width=900,
    height=500
)
fig.show()


### Results and Interpretation (9-Model ARIMA + Hybrid System)

#### 1. Why Lasso and the ARIMA+Lasso Hybrid Outperform
- **Multicollinearity Resolution**: Exogenous features like Crude Oil, DXY, and US-India Interest spreads are highly correlated. Standard ARIMAX suffers from coefficient instability. **Lasso (L1 Regularization)** penalizes coefficients, driving redundant features to zero, yielding more robust out-of-sample forecasts.
- **Result**: Standalone Lasso achieves the lowest RMSE (0.3973) and Theil's U (0.9930) of any standalone model, with a 59.5% MDA. The ARIMA+Lasso Hybrid pushes MDA further to 59.0% and the highest Sharpe Ratio (Rf=0) of any model. See model_metrics.csv for exact current numbers since the rolling test window shifts as new data is collected.

#### 2. ARIMA vs. the Old Seasonal SARIMA Baseline
- Non-seasonal ARIMA(1,1,0) outperforms the retired seasonal SARIMA baseline, supported by the seasonal audit in notebook 02b_seasonal_analysis.ipynb showing no significant ACF/PACF spikes at lags 13/26/52.
- Adding macro indicators (ARIMAX) does not reliably beat plain ARIMA once look-ahead bias is removed -- this is consistent with the Meese-Rogoff finding that naive/autoregressive baselines are hard to beat with raw fundamentals alone.

#### 3. Tree-Based Models Underperform Linear/Regularized Models
- Random Forest and Gradient Boosting post higher RMSE and lower MDA than Lasso or ARIMA on this dataset. With only ~400 training weeks and 8 features, the tree ensembles appear to overfit noise rather than capture genuine signal -- this is a real limitation worth stating plainly rather than hiding.

*Note: exact metric values shift slightly each time the pipeline is re-run, because collect_data.py always pulls data up to today's date. Treat the numbers above as illustrative of the ranking/pattern; see data/processed/model_metrics.csv (or the frozen benchmark snapshot, if present) for the authoritative numbers as of a specific date.*


## Next-Week Out-of-Sample Signal Generation

Below we train the selected model (**ARIMA+Lasso Hybrid** -- chosen because it posts the best Sharpe Ratio and MDA of any model in the backtest above) on the **entire dataset** to generate the directional forecast and signal for the upcoming week. This matches the logic implemented in `generate_next_week_signal.py`.


In [9]:
# 1. Fit ARIMA on all historical data
X = model_df[features_list]
y = model_df["USDINR_diff"]

print("Finding optimal ARIMA order on whole dataset...")
auto_model_all = auto_arima(
    y,
    seasonal=False,
    stepwise=True,
    suppress_warnings=True,
    max_p=5, max_q=5
)
order_all = auto_model_all.order
print(f"Optimal ARIMA order on full dataset: {order_all}")

model_arima_all = SARIMAX(
    y,
    order=order_all,
    seasonal_order=(0,0,0,0),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

pred_arima_diff_all = model_arima_all.forecast(steps=1).iloc[0]

# Compute in-sample residuals of ARIMA
arima_fitted_all = model_arima_all.fittedvalues
arima_residuals_all = y - arima_fitted_all

# 2. Construct features for the NEXT week (t_last + 1)
last_date = weekly.index[-1]
next_week_date = last_date + pd.Timedelta(weeks=1)

# Base features at t_last (lagged by 1 week relative to next week)
next_crude_diff_lag1 = weekly["CRUDE_diff"].iloc[-1]
next_dxy_diff_lag1 = weekly["DXY_diff"].iloc[-1]
next_rate_spread_diff_lag1 = weekly["Rate_Spread_diff"].iloc[-1]
next_geo_tension_lag1 = weekly["Geo_Tension"].iloc[-1]

# Momentum at t_last (known at last_date)
next_inr_mom_4w = weekly["USDINR_diff"].iloc[-4:].mean()
next_inr_mom_12w = weekly["USDINR_diff"].iloc[-12:].mean()

# Calendar dummies for next week (t_last + 1)
next_is_fiscal_yr_end = float(next_week_date.month == 3)
next_is_qtr_end = float(next_week_date.month in [3, 6, 9, 12])

# Build next week's feature vector
X_next = pd.DataFrame([{
    "CRUDE_diff_lag1": next_crude_diff_lag1,
    "DXY_diff_lag1": next_dxy_diff_lag1,
    "Rate_Spread_diff_lag1": next_rate_spread_diff_lag1,
    "Geo_Tension_lag1": next_geo_tension_lag1,
    "inr_mom_4w": next_inr_mom_4w,
    "inr_mom_12w": next_inr_mom_12w,
    "is_fiscal_yr_end": next_is_fiscal_yr_end,
    "is_qtr_end": next_is_qtr_end
}])

# 3. Fit Gradient Boosting on X to predict the residuals
model_lasso_resid_all = Lasso(alpha=0.001).fit(X, arima_residuals_all)
pred_lasso_resid_all = model_lasso_resid_all.predict(X_next)[0]

# 4. Hybrid prediction
pred_diff_all = pred_arima_diff_all + pred_lasso_resid_all
current_rate = weekly["USDINR"].iloc[-1]
pred_rate = current_rate + pred_diff_all

signal = "STRENGTHEN" if pred_diff_all < 0 else "WEAKEN"
change_paise = abs(pred_diff_all) * 100

output = {
    "as_of_date": last_date.strftime("%Y-%m-%d"),
    "next_week_date": next_week_date.strftime("%Y-%m-%d"),
    "current_rate": float(current_rate),
    "predicted_change": float(pred_diff_all),
    "predicted_rate": float(pred_rate),
    "signal": signal,
    "change_paise": float(change_paise),
    "model_type": "ARIMA+Lasso_Hybrid"
}

print("--- NEXT WEEK FORECAST SIGNAL ---")
for k, v in output.items():
    print(f"{k}: {v}")

Finding optimal ARIMA order on whole dataset...


Optimal ARIMA order on full dataset: (0, 0, 1)
--- NEXT WEEK FORECAST SIGNAL ---
as_of_date: 2026-05-24
next_week_date: 2026-05-31
current_rate: 96.30169982910157
predicted_change: 0.02587251543009153
predicted_rate: 96.32757234453166
signal: WEAKEN
change_paise: 2.587251543009153
model_type: ARIMA+GradientBoosting_Hybrid
